<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/Day_7_The_Lying_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
X, y = make_classification(
n_samples=2000, n_features=8, n_informative=4, n_redundant=1,
weights=[0.92, 0.08], flip_y=0.03, class_sep=0.85, random_state=7,
)
columns = ["income", "loan_amount", "age", "months_employed",
"existing_loans", "credit_score", "debt_to_income", "late_payments_12m"]
df = pd.DataFrame(X, columns=columns)
df["defaulted"] = y
print(df.shape)
print(df["defaulted"].value_counts())
print("default rate:", round(df["defaulted"].mean(), 3))

(2000, 9)
defaulted
0    1813
1     187
Name: count, dtype: int64
default rate: 0.094


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=42, stratify=y
)
print(len(X_train), "training rows |", len(X_test), "test rows")
print("defaulters in test set:", y_test.sum())

1500 training rows | 500 test rows
defaulters in test set: 47


In [6]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print("dummy accuracy:", round(dummy.score(X_test, y_test), 3))

dummy accuracy: 0.906


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
print("model accuracy:", round(model.score(X_test, y_test), 3))

model accuracy: 0.92


In [10]:
from sklearn.metrics import confusion_matrix, classification_report
preds = model.predict(X_test)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=["repaid", "default"], digits=2))

[[451   2]
 [ 38   9]]
              precision    recall  f1-score   support

      repaid       0.92      1.00      0.96       453
     default       0.82      0.19      0.31        47

    accuracy                           0.92       500
   macro avg       0.87      0.59      0.63       500
weighted avg       0.91      0.92      0.90       500



In [12]:
balanced = make_pipeline(
StandardScaler(),
LogisticRegression(max_iter=1000, class_weight="balanced")
)
balanced.fit(X_train, y_train)
preds_b = balanced.predict(X_test)
print("accuracy:", round(balanced.score(X_test, y_test), 3))
print(classification_report(y_test, preds_b, target_names=["repaid", "default"], digits=2))

accuracy: 0.776
              precision    recall  f1-score   support

      repaid       0.95      0.79      0.86       453
     default       0.24      0.64      0.35        47

    accuracy                           0.78       500
   macro avg       0.60      0.71      0.61       500
weighted avg       0.89      0.78      0.82       500



In [14]:
from sklearn.metrics import precision_score, recall_score
probabilities = model.predict_proba(X_test)[:, 1]
print(f"{'threshold':>10} {'recall':>8} {'precision':>10} {'flagged':>8}")
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    preds_t = (probabilities >= t).astype(int)
    print(f"{t:>10} {recall_score(y_test, preds_t):>8.2f} "
          f"{precision_score(y_test, preds_t, zero_division=0):>10.2f} {preds_t.sum():>8}")

 threshold   recall  precision  flagged
       0.5     0.19       0.82       11
       0.4     0.26       0.67       18
       0.3     0.32       0.50       30
       0.2     0.45       0.45       47
       0.1     0.57       0.23      118
      0.05     0.70       0.17      195


In [16]:
from sklearn.tree import DecisionTreeClassifier
deep = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print("deep tree — train:", round(deep.score(X_train, y_train), 3),
"| test:", round(deep.score(X_test, y_test), 3))
shallow = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print("depth-3 — train:", round(shallow.score(X_train, y_train), 3),
"| test:", round(shallow.score(X_test, y_test), 3))

deep tree — train: 1.0 | test: 0.902
depth-3 — train: 0.943 | test: 0.922


In [17]:
test_df = pd.DataFrame(X_test, columns=columns)
test_df["actual"] = y_test
test_df["predicted"] = balanced.predict(X_test)
test_df["probability"] = balanced.predict_proba(X_test)[:, 1]
missed = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 0)]
print(len(missed), "defaulters missed")
print(missed.sort_values("probability", ascending=False).head(10))

17 defaulters missed
       income  loan_amount       age  months_employed  existing_loans  \
335  2.044454     0.209550  0.069818        -0.326909        0.350912   
116 -0.269323     1.299351 -0.288072        -3.399208        2.028629   
433  0.068840     1.376970 -0.016365        -3.587908       -1.686218   
486 -0.578013     0.097075  0.598651        -0.214851       -1.057215   
58   1.764601     0.477137  0.815408        -1.811411       -1.101770   
248 -0.410474     1.099338  1.552991         0.810463       -1.823105   
179 -0.346579    -1.508905  0.417276         0.028676        1.262860   
167 -1.874132    -1.277142  0.814503        -0.326278       -0.533810   
497 -0.735079    -0.551122  1.498643         1.721922        1.649005   
323 -0.364610    -0.370659  0.885062        -1.269634        0.562126   

     credit_score  debt_to_income  late_payments_12m  actual  predicted  \
335      2.241093       -0.672653           1.818542       1          0   
116      0.829835       -

In [19]:
caught = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 1)]
comparison = pd.DataFrame({"missed": missed[columns].mean(),
"caught": caught[columns].mean()})
comparison["difference"] = comparison["missed"] - comparison["caught"]
print(comparison.round(2))

                   missed  caught  difference
income              -0.19   -0.01       -0.18
loan_amount          0.13   -0.03        0.16
age                  1.04   -1.08        2.12
months_employed     -0.76    0.06       -0.83
existing_loans       0.14   -0.01        0.15
credit_score        -0.10    0.23       -0.33
debt_to_income      -0.11    0.18       -0.29
late_payments_12m    0.93   -0.47        1.41
